# 건축공학 에이전트 시스템 — Structural Engineering Agents

**도메인 실습: 구조 검토 시스템**

이 노트북에서 다루는 내용:
1. **v1** 병렬화: 휨/전단/사용성을 동시에 검토
2. **v2** 체이닝: 부재 데이터 추출 → 기준 검토 → 보고서 생성
3. **v3** 에이전트: 자율적으로 KDS 기준을 조회하고 검토하는 에이전트

**입력**: 부재 제원 (b, d, fck, fy, As, Av, s)

In [ ]:
# ── Setup ──────────────────────────────────────────────
import anthropic
import asyncio
import json
from dotenv import load_dotenv

load_dotenv()

client = anthropic.Anthropic()
async_client = anthropic.AsyncAnthropic()
MODEL = "claude-haiku-4-5"

In [ ]:
# RC 보 부재 제원
member_data = {
    "type": "RC Beam",
    "b": 400,      # 폭 (mm)
    "d": 550,      # 유효깊이 (mm)
    "h": 600,      # 전체깊이 (mm)
    "fck": 30,     # 콘크리트 압축강도 (MPa)
    "fy": 400,     # 철근 항복강도 (MPa)
    "As": 2580,    # 인장철근 단면적 (mm^2) — 6-D25 (=6x430)
    "Av": 142,     # 전단철근 단면적 (mm^2) — D10 2-leg
    "s": 200,      # 전단철근 간격 (mm)
    "L": 8000,     # 경간 (mm)
    "Mu": 450,     # 작용 휨모멘트 (kN·m)
    "Vu": 280,     # 작용 전단력 (kN)
}

member_str = json.dumps(member_data, indent=2, ensure_ascii=False)
print("부재 제원:")
print(member_str)

---
## v1. 병렬화 — 휨/전단/사용성 동시 검토

세 가지 검토를 **동시에** 실행하여 시간을 절약합니다.

In [ ]:
review_perspectives = [
    {
        "name": "휨 검토",
        "system": (
            "You are a structural engineer specializing in flexural design. "
            "Review the RC beam for flexural adequacy per KDS 14 20 20. "
            "Calculate phi*Mn and compare with Mu. "
            "Use phi=0.85 for flexure. Show calculations. Respond in Korean."
        )
    },
    {
        "name": "전단 검토",
        "system": (
            "You are a structural engineer specializing in shear design. "
            "Review the RC beam for shear adequacy per KDS 14 20 22. "
            "Calculate Vc, Vs, and phi*Vn. Compare with Vu. "
            "Use phi=0.75 for shear. Show calculations. Respond in Korean."
        )
    },
    {
        "name": "사용성 검토",
        "system": (
            "You are a structural engineer specializing in serviceability. "
            "Check the beam for deflection limits (L/360 for live load, "
            "L/240 for total load) and crack width limits per KDS 14 20 30. "
            "Respond in Korean."
        )
    },
]

async def review_perspective(perspective, data_str):
    """단일 관점으로 부재를 비동기 검토한다."""
    response = await async_client.messages.create(
        model=MODEL,
        max_tokens=2048,
        system=perspective["system"],
        messages=[{"role": "user", "content": f"Review this member:\n{data_str}"}]
    )
    return {"name": perspective["name"], "result": response.content[0].text}

import time
start = time.time()
results = await asyncio.gather(
    *[review_perspective(p, member_str) for p in review_perspectives]
)
elapsed = time.time() - start

print(f"병렬 검토 완료: {elapsed:.1f}초\n")
for r in results:
    print(f"{'='*50}")
    print(f"📋 {r['name']}")
    print(f"{'='*50}")
    print(r["result"])
    print()

---
## v2. 체이닝 — 추출 → 검토 → 보고서

3단계 순차 파이프라인으로 구조적인 검토 보고서를 생성합니다.

In [ ]:
def chain_step(system_prompt, user_message):
    response = client.messages.create(
        model=MODEL, max_tokens=2048,
        system=system_prompt,
        messages=[{"role": "user", "content": user_message}]
    )
    return response.content[0].text

# Step 1: 부재 데이터 정리 및 핵심 파라미터 추출
print("Step 1: 핵심 파라미터 추출...")
extracted = chain_step(
    system_prompt=(
        "You are a structural engineer. Extract and organize the key "
        "design parameters from the member data. Calculate derived values "
        "like rho (reinforcement ratio), d/b ratio, etc. Respond in Korean."
    ),
    user_message=f"부재 데이터:\n{member_str}"
)
print(f"  완료 ({len(extracted)} 글자)")

# Step 2: KDS 기준 검토 (Step 1 출력이 입력)
print("Step 2: KDS 기준 검토...")
review = chain_step(
    system_prompt=(
        "You are a structural code compliance reviewer. "
        "Based on the extracted parameters, check compliance with "
        "KDS 14 20 (RC design code). Check: min/max reinforcement ratio, "
        "flexural capacity, shear capacity, spacing requirements. "
        "Respond in Korean."
    ),
    user_message=f"추출된 파라미터:\n{extracted}"
)
print(f"  완료 ({len(review)} 글자)")

# Step 3: 검토 보고서 생성 (Step 2 출력이 입력)
print("Step 3: 보고서 생성...")
report = chain_step(
    system_prompt=(
        "You are a senior structural engineer writing a review report. "
        "Create a formal structural review report with: "
        "(1) Summary of compliance status, "
        "(2) Detailed findings, "
        "(3) Recommendations for any non-compliant items. "
        "Use professional engineering report format. Respond in Korean."
    ),
    user_message=f"기준 검토 결과:\n{review}"
)
print(f"  완료 ({len(report)} 글자)")

print("\n" + "=" * 50)
print("📊 구조 검토 보고서")
print("=" * 50)
print(report)

---
## v3. 에이전트 — 자율 구조 검토

에이전트에게 구조 검토 도구를 제공하고, **자율적으로** 검토를 수행하게 합니다.

**도전 과제**: 아래 도구 정의와 에이전틱 루프를 완성하세요.

In [ ]:
import math

# 구조 검토 도구 함수들
def check_flexure(b, d, fck, fy, As, Mu):
    """휨 검토: phi*Mn vs Mu"""
    a = As * fy / (0.85 * fck * b)
    Mn = As * fy * (d - a / 2) / 1e6  # kN·m
    phi_Mn = 0.85 * Mn
    ratio = phi_Mn / Mu
    status = "OK" if ratio >= 1.0 else "NG"
    return json.dumps({
        "phi_Mn_kNm": round(phi_Mn, 1),
        "Mu_kNm": Mu,
        "ratio": round(ratio, 3),
        "status": status
    })

def check_shear(b, d, fck, fy, Av, s, Vu):
    """전단 검토: phi*Vn vs Vu"""
    Vc = (1/6) * math.sqrt(fck) * b * d / 1000  # kN
    Vs = Av * fy * d / s / 1000  # kN
    Vn = Vc + Vs
    phi_Vn = 0.75 * Vn
    ratio = phi_Vn / Vu
    status = "OK" if ratio >= 1.0 else "NG"
    return json.dumps({
        "Vc_kN": round(Vc, 1),
        "Vs_kN": round(Vs, 1),
        "phi_Vn_kN": round(phi_Vn, 1),
        "Vu_kN": Vu,
        "ratio": round(ratio, 3),
        "status": status
    })

def check_reinforcement_ratio(b, d, fck, fy, As):
    """철근비 검토: rho_min <= rho <= rho_max"""
    rho = As / (b * d)
    rho_min = max(0.25 * math.sqrt(fck) / fy, 1.4 / fy)
    rho_max = 0.85 * 0.85 * fck / fy * 600 / (600 + fy) * 0.75
    status = "OK" if rho_min <= rho <= rho_max else "NG"
    return json.dumps({
        "rho": round(rho, 5),
        "rho_min": round(rho_min, 5),
        "rho_max": round(rho_max, 5),
        "status": status
    })

# 테스트
print("휨 검토:", check_flexure(**{k: member_data[k] for k in ['b','d','fck','fy','As','Mu']}))
print("전단 검토:", check_shear(**{k: member_data[k] for k in ['b','d','fck','fy','Av','s','Vu']}))
print("철근비 검토:", check_reinforcement_ratio(**{k: member_data[k] for k in ['b','d','fck','fy','As']}))

In [ ]:
# TODO: 도구 스키마를 정의하세요
structural_tools = [
    # {
    #     "name": "check_flexure",
    #     "description": "...",
    #     "input_schema": { ... }
    # },
    # {
    #     "name": "check_shear",
    #     ...
    # },
    # {
    #     "name": "check_reinforcement_ratio",
    #     ...
    # }
]

# TODO: 도구 실행 라우터
# def execute_structural_tool(name, input_data):
#     if name == "check_flexure":
#         return check_flexure(**input_data)
#     ...

# TODO: 에이전틱 루프
# def run_structural_agent(member_data, max_turns=10):
#     ...

---
## 제출 체크리스트

- [ ] v1: 병렬화 — 3가지 검토 동시 실행 + 시간 측정
- [ ] v2: 체이닝 — 3단계 파이프라인 + 보고서 생성
- [ ] v3: 에이전트 — 도구 스키마 + 에이전틱 루프 + 자율 검토 실행
- [ ] (보너스) v3에 InspectableAgent 패턴 적용